In [6]:
import pandas as pd
import json

def extract_conversations(csv_file):
    # Read the CSV file
    df = pd.read_csv(csv_file, sep='\t')
    
    # Dictionary to store all conversations
    all_conversations = []
    
    # Group by dialog_id to process each conversation
    for dialog_id in df['dialog_id'].unique():
        dialog_df = df[df['dialog_id'] == dialog_id].sort_values('utt_id')
        
        conversation = []
        current_speaker = None
        current_content = []
        
        for _, row in dialog_df.iterrows():
            speaker = row['speaker']
            text = row['text']
            
            # If same speaker continues, accumulate text
            if speaker == current_speaker:
                current_content.append(text)
            else:
                # If we have accumulated content, add it to conversation
                if current_content and current_speaker is not None:
                    prev_role = 'user' if current_speaker == 'SEEKER' else 'assistant'
                    conversation.append({
                        'role': prev_role,
                        'content': ' '.join(current_content)
                    })
                
                # Start new speaker's content
                current_speaker = speaker
                current_content = [text]
        
        # Don't forget the last accumulated content
        if current_content and current_speaker is not None:
            role = 'user' if current_speaker == 'SEEKER' else 'assistant'
            conversation.append({
                'role': role,
                'content': ' '.join(current_content)
            })

        # Add conversation to the list if it has content
        if conversation:
            all_conversations.append({
                'dialog_id': dialog_id,
                'ground_truth': df['movie_id'][0],
                'conversation': conversation
            })
    
    return all_conversations

def save_conversations_to_json(conversations, output_file):
    # Save to JSON file
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(conversations, f, indent=2, ensure_ascii=False)

# Main execution
if __name__ == "__main__":
    # Extract conversations from CSV
    conversations = extract_conversations('raw/train.tsv')
    
    # Save to JSON file
    save_conversations_to_json(conversations, 'multiturn_form/train.json')
    
    print(f"Extracted {len(conversations)} conversations and saved to conversations.json")
    
    # Print a sample conversation for verification
    if conversations:
        print("\nSample conversation:")
        sample_conv = conversations[0]['conversation']
        for i, turn in enumerate(sample_conv[:4]):  # Show first 4 turns
            print(f"{turn['role']}: {turn['content']}")

Extracted 801 conversations and saved to conversations.json

Sample conversation:
assistant: Hi There! What types of movies do you like to watch?
user: Hello! I'm more of an action movie or a good romance and mystery movie.
assistant: I just saw the trailer for Knives Out when I went to see Joker and it looked like a good mix of action and mystery!
user: I seen that one too as I seen Joker about a month ago. I thought about asking my fiance about going and seeing it.


In [3]:
import os

# Get the current working directory
current_directory = os.getcwd()

# Print the current working directory
print(current_directory)

/home/sagemaker-user/csbai/multiturn_rl
